# GraphMS-Net — Reproducibility and Patient-Level Inference Notebook

This notebook executes the **frozen GraphMS v3.5.1 Hybrid research pipeline** from a clean GitHub clone in a CUDA-enabled Google Colab environment.

The default reproducibility demonstration is **zero-upload**: the attributed public `MSLesSeg_P10_T1` FLAIR/T1/T2 triplet is bundled with the repository, allowing the complete workflow to be executed with **Run all** without manual MRI selection.

The notebook performs four distinct tasks:

1. verifies the frozen repository and runtime contracts;
2. presents the complete committed quantitative research record for Stages 12–16;
3. executes the frozen patient-level inference pipeline;
4. verifies the resulting provenance and accepted demonstration output.

**Scientific scope:** segmentation performance is reported from the frozen **development five-fold cross-validation** evaluation. Stage13 metrics are development patient-grouped CV results. The notebook does not present these results as independent external validation or clinical-performance evidence.

## 0. Before running

In Colab choose **Runtime → Change runtime type → GPU**.

For the accepted reproducibility demonstration, use:
- case ID: `MSLesSeg_P10_T1`
- `*_0000.nii.gz` = FLAIR
- `*_0001.nii.gz` = T1
- `*_0002.nii.gz` = T2

For a case outside the frozen 93-case development registry, an explicit fold `0..4` is required because no new-patient ensemble rule was validated.

In [ ]:
import subprocess, sys

print("Checking NVIDIA GPU...")
subprocess.run(["nvidia-smi"], check=True)

## 1. Clone the frozen repository

This always starts from the public `main` branch so the notebook does not depend on a local copy.

In [ ]:
from pathlib import Path
import os, shutil, subprocess

REPO = Path("/content/GraphMS-Net")
if REPO.exists():
    shutil.rmtree(REPO)

subprocess.run(
    ["git", "clone", "--branch", "main", "--single-branch",
     "https://github.com/sath17-o/GraphMS-Net.git", str(REPO)],
    check=True,
)

os.chdir(REPO)
print("Repository:", REPO)
subprocess.run(["git", "log", "-1", "--oneline"], check=True)

## 2. Install the accepted inference environment

The accepted CUDA environment uses **PyTorch 2.8.0 / CUDA 12.6** and `nnunetv2==2.8.1`.

The notebook installs the CUDA PyTorch wheel first, then the repository's frozen inference and verification dependencies.

In [ ]:
import subprocess, sys

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "torch==2.8.0", "torchvision==0.23.0",
    "--index-url", "https://download.pytorch.org/whl/cu126"
], check=True)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "-r", "requirements-inference.txt"
], check=True)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "-r", "requirements-verify.txt"
], check=True)

print("Environment installation complete.")

## 3. Verify the CUDA neural runtime

The full 3-D GraphMS patient path is intentionally CUDA-only. If this cell fails, stop and fix the Colab GPU runtime before continuing.

In [ ]:
import subprocess, sys

subprocess.run(
    [sys.executable, "scripts/check_neural_runtime.py"],
    check=True,
)

## 4. Verify the frozen research package

These checks do **not** retrain or re-select the model.

They verify the frozen manifests/audits and replay the committed Stage16 five-fold aggregation from the 93-case development evaluation table.

In [ ]:
import subprocess, sys

print("\n=== Frozen package verification ===")
subprocess.run(
    [sys.executable, "scripts/run_pipeline.py", "--mode", "verify"],
    check=True,
)

print("\n=== Stage16 frozen evaluation replay ===")
subprocess.run(
    [sys.executable, "scripts/run_pipeline.py", "--mode", "evaluation-replay"],
    check=True,
)

## 5. Frozen quantitative research record

The following tables are loaded directly from the committed frozen result artifacts. No values are retyped or recomputed from the live demonstration.

This section distinguishes three forms of evidence:

- **Stage16 segmentation evaluation:** five-fold development segmentation metrics using frozen ground truth.
- **Stage13 downstream modeling:** patient-grouped development CV for EDSS≥4 classification and EDSS regression.
- **Stages12–16 integrity/provenance:** audit completion, frozen training configuration, task status, and acceptance evidence.

The live patient-level run later in this notebook does **not** use ground truth.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
from IPython.display import display, Markdown, HTML

RESULTS = REPO / "results"
EVIDENCE = REPO / "evidence" / "acceptance"

def load_json(path):
    return json.loads(Path(path).read_text())

stage12_manifest = load_json(RESULTS / "stage12" / "STAGE12_CANONICAL_FEATURE_MANIFEST.json")
stage13_manifest = load_json(RESULTS / "stage13" / "STAGE13_FINAL_MANIFEST.json")
stage14_manifest = load_json(RESULTS / "stage14" / "STAGE14_FINAL_MANIFEST.json")
stage15_manifest = load_json(RESULTS / "stage15" / "STAGE15_FINAL_MANIFEST.json")
stage16_manifest = load_json(RESULTS / "stage16" / "STAGE16_FINAL_MANIFEST.json")

stage16_summary = pd.read_csv(RESULTS / "stage16" / "STAGE16_5FOLD_MEAN_STD.csv")
stage16_per_fold = pd.read_csv(RESULTS / "stage16" / "STAGE16_PER_FOLD_METRICS.csv")
stage16_per_case = pd.read_csv(RESULTS / "stage16" / "STAGE16_PER_CASE_METRICS.csv")

print("Frozen result artifacts loaded successfully.")

### 5.1 Research population, protocol identity, and frozen configuration

In [ ]:
population = pd.DataFrame([
    ["Development MRI scans", stage13_manifest["development"]["scans"]],
    ["Development patients", stage13_manifest["development"]["patients"]],
    ["Unique development cases", stage13_manifest["development"]["unique_cases"]],
    ["EDSS≥4 scans", stage13_manifest["development"]["edss_ge4_scans"]],
    ["EDSS≥4 patients", stage13_manifest["development"]["edss_ge4_patients"]],
    ["Stage16 evaluated cases", stage16_manifest["n_cases"]],
    ["Stage16 outer folds", stage16_manifest["n_folds"]],
    ["Fold counts", str(stage16_manifest["fold_counts"])],
    ["Segmentation claim scope", stage16_manifest["claim_scope"]],
    ["Stage13 claim scope", stage13_manifest["claim_scope"]],
], columns=["Item", "Frozen value"])

identity = pd.DataFrame([
    ["Final segmentation model", stage16_manifest["final_model"]],
    ["Protocol SHA", stage16_manifest["protocol_sha"]],
    ["Fusion implementation", stage16_manifest["fusion_implementation_id"]],
    ["Stage11", stage16_manifest["stage11"]],
    ["Stage12 schema", stage13_manifest["stage12_schema"]],
    ["Stage13 classifier", stage13_manifest["primary_classification"]["family"]],
    ["Stage13 regressor", stage13_manifest["primary_regression"]["family"]],
    ["Stage14 multi-task status", stage14_manifest["multitask_branch_status"]],
], columns=["Component", "Frozen identity"])

training = pd.DataFrame([
    ["Segmentation loss provenance", stage15_manifest["stage10_loss_provenance"]["loss"]],
    ["Optimizer", stage15_manifest["selected_optimizer"]],
    ["Learning-rate scheduler", stage15_manifest["selected_scheduler"]],
    ["Dropout", stage15_manifest["dropout"]],
    ["Weight decay", stage15_manifest["weight_decay"]],
    ["Final selected model", stage15_manifest["final_model"]],
    ["Current pipeline jointly retrained", stage15_manifest["current_pipeline_is_jointly_retrained"]],
    ["New training performed in packaging", stage15_manifest["new_training_performed"]],
], columns=["Training/provenance item", "Frozen value"])

display(Markdown("**Development population and claim scope**"))
display(population)
display(Markdown("**Frozen model identity**"))
display(identity)
display(Markdown("**Frozen training configuration / provenance**"))
display(training)

### 5.2 Stage12–16 audit completeness

These audits are fail-closed repository checks. A PASS count indicates that every committed gate for that stage passed; it is not an additional performance metric.

In [ ]:
audit_paths = {
    "Stage12 — lesion feature extraction": RESULTS / "stage12" / "STAGE12_FINAL_AUDIT.csv",
    "Stage13 — risk/EDSS modeling": RESULTS / "stage13" / "STAGE13_FINAL_AUDIT.csv",
    "Stage14 — multi-task evidence": RESULTS / "stage14" / "STAGE14_FINAL_AUDIT.csv",
    "Stage15 — training/configuration": RESULTS / "stage15" / "STAGE15_FINAL_AUDIT.csv",
    "Stage16 — evaluation metrics": RESULTS / "stage16" / "STAGE16_FINAL_AUDIT.csv",
}

audit_rows = []
for stage, path in audit_paths.items():
    df = pd.read_csv(path)
    passed = int((df["status"] == "PASS").sum())
    total = len(df)
    audit_rows.append([stage, passed, total, "PASS" if passed == total else "FAIL"])

audit_summary = pd.DataFrame(
    audit_rows,
    columns=["Stage", "PASS gates", "Total gates", "Overall status"],
)
display(audit_summary)

assert (audit_summary["Overall status"] == "PASS").all()
print("All committed Stage12–16 audit gates PASS.")

### 5.3 Stage16 segmentation performance — five-fold aggregate

Aggregation is the **equal-weight mean of the five outer-fold case means**, with sample standard deviation across the five folds (`ddof=1`). HD95 is reported in millimetres.

In [ ]:
segmentation_summary = stage16_summary[
    ["Metric", "5Fold_Mean", "5Fold_Std", "Mean_PlusMinus_Std", "aggregation"]
].copy()

display(segmentation_summary)

### 5.4 Stage16 segmentation performance — per fold

In [ ]:
display(stage16_per_fold)

print("Fold counts:", stage16_manifest["fold_counts"])

### 5.5 Stage16 segmentation performance — complete 93-case record

The table below contains the complete committed case-level evaluation metrics. It is frozen evaluation evidence and uses the development ground-truth masks. The later live inference demonstration does not use these ground-truth masks.

In [ ]:
case_metric_columns = [
    "case", "fold", "DSC", "IoU", "Sensitivity", "Specificity", "HD95_mm",
    "TP", "FP", "FN", "TN",
]
case_metrics = stage16_per_case[case_metric_columns].copy()

case_descriptive = case_metrics[
    ["DSC", "IoU", "Sensitivity", "Specificity", "HD95_mm"]
].describe(percentiles=[0.25, 0.50, 0.75]).T

display(Markdown("**Descriptive statistics across all 93 development cases**"))
display(case_descriptive)

display(Markdown("**Complete 93-case metric table**"))
display(case_metrics)

assert len(case_metrics) == 93
assert case_metrics["case"].nunique() == 93

### 5.6 Stage16 fold-level visual summary

In [ ]:
import matplotlib.pyplot as plt

ax = stage16_per_fold.plot(
    x="fold",
    y=["DSC", "IoU", "Sensitivity", "Specificity"],
    marker="o",
    figsize=(9, 5),
)
ax.set_title("GraphMS v3.5.1 Hybrid — segmentation metrics by outer fold")
ax.set_xlabel("Outer fold")
ax.set_ylabel("Metric value")
ax.set_ylim(0.0, 1.05)
ax.grid(True, alpha=0.25)
plt.show()

ax = stage16_per_fold.plot(
    x="fold",
    y="HD95_mm",
    marker="o",
    figsize=(9, 4),
    legend=False,
)
ax.set_title("GraphMS v3.5.1 Hybrid — HD95 by outer fold")
ax.set_xlabel("Outer fold")
ax.set_ylabel("HD95 (mm)")
ax.grid(True, alpha=0.25)
plt.show()

### 5.7 Stage13 EDSS≥4 classification — complete frozen development metrics

The classifier is the frozen **MRI_SPATIAL_SVM (C=30)** using the `mri_spatial` feature set. Metrics are development patient-grouped CV/OOF results and are not external-test estimates.

In [ ]:
cls = stage13_manifest["primary_classification"]
nested_cls = cls["nested_cv"]

classification_nested = pd.DataFrame([
    ["ROC-AUC mean", nested_cls["nested_auc_mean"]],
    ["ROC-AUC SD", nested_cls["nested_auc_std"]],
    ["ROC-AUC median", nested_cls["nested_auc_median"]],
    ["Average precision mean", nested_cls["nested_ap_mean"]],
    ["Average precision SD", nested_cls["nested_ap_std"]],
    ["Outer fold results", nested_cls["folds"]],
], columns=["Nested-CV metric", "Value"])

balanced = cls["fixed_oof_balanced"]
highsens = cls["fixed_oof_high_sensitivity"]

classification_operating_points = pd.DataFrame([
    {
        "Operating point": "Balanced",
        "Threshold": cls["thresholds"]["balanced_threshold"],
        "ROC-AUC": balanced["roc_auc"],
        "Average precision": balanced["average_precision"],
        "Brier score": balanced["brier"],
        "Accuracy": balanced["accuracy"],
        "Balanced accuracy": balanced["balanced_accuracy"],
        "Sensitivity": balanced["sensitivity"],
        "Specificity": balanced["specificity"],
        "Precision": balanced["precision"],
        "F1": balanced["f1"],
    },
    {
        "Operating point": "High sensitivity",
        "Threshold": cls["thresholds"]["high_sensitivity_threshold"],
        "ROC-AUC": highsens["roc_auc"],
        "Average precision": highsens["average_precision"],
        "Brier score": highsens["brier"],
        "Accuracy": highsens["accuracy"],
        "Balanced accuracy": highsens["balanced_accuracy"],
        "Sensitivity": highsens["sensitivity"],
        "Specificity": highsens["specificity"],
        "Precision": highsens["precision"],
        "F1": highsens["f1"],
    },
])

display(Markdown("**Repeated nested patient-grouped CV**"))
display(classification_nested)
display(Markdown("**Fixed-model out-of-fold operating points**"))
display(classification_operating_points)

print("High-sensitivity target:", cls["thresholds"]["high_sensitivity_target"])

### 5.8 Stage13 EDSS regression — complete frozen development metrics

The regressor is the frozen **MRI_SPATIAL_RIDGE (alpha=30)** using the `mri_spatial` feature set.

In [ ]:
reg = stage13_manifest["primary_regression"]
nested_reg = reg["nested_cv"]
fixed_reg = reg["fixed_oof"]

regression_nested = pd.DataFrame([
    ["RMSE mean", nested_reg["nested_rmse_mean"]],
    ["RMSE SD", nested_reg["nested_rmse_std"]],
    ["RMSE median", nested_reg["nested_rmse_median"]],
    ["MAE mean", nested_reg["nested_mae_mean"]],
    ["MAE SD", nested_reg["nested_mae_std"]],
    ["R² mean", nested_reg["nested_r2_mean"]],
    ["Spearman rho mean", nested_reg["nested_spearman_mean"]],
    ["Outer fold results", nested_reg["folds"]],
], columns=["Nested-CV metric", "Value"])

regression_fixed = pd.DataFrame([
    ["MAE", fixed_reg["mae"]],
    ["RMSE", fixed_reg["rmse"]],
    ["R²", fixed_reg["r2"]],
    ["Spearman rho", fixed_reg["spearman_rho"]],
], columns=["Fixed OOF metric", "Value"])

display(Markdown("**Repeated nested patient-grouped CV**"))
display(regression_nested)
display(Markdown("**Fixed-model out-of-fold metrics**"))
display(regression_fixed)

### 5.9 Stage11 post-processing recipes and Stage12 feature-extraction integrity

In [ ]:
recipes = stage12_manifest["stage11_cross_fitted_recipes"]
stage11_table = pd.DataFrame([
    {
        "Fold": int(fold),
        "Probability threshold": cfg["threshold"],
        "Minimum voxels": cfg["minvox"],
        "Connectivity": cfg["connectivity"],
        "Morphology": cfg["morphology"],
    }
    for fold, cfg in recipes.items()
]).sort_values("Fold")

stage12_integrity = pd.DataFrame([
    ["Cases", stage12_manifest["n_cases"]],
    ["Patients", stage12_manifest["n_patients"]],
    ["Feature schema", stage12_manifest["feature_schema_version"]],
    ["Atlas/spatial context", stage12_manifest["atlas_spatial_context"]],
    ["Connectivity", stage12_manifest["connectivity"]],
    ["Morphology", stage12_manifest["morphology"]],
    ["Ground truth used for feature extraction", stage12_manifest["ground_truth_mask_used_for_features"]],
    ["Stage13 feature contract satisfied", stage12_manifest["stage13_feature_contract_satisfied"]],
], columns=["Stage12 item", "Frozen value"])

display(Markdown("**Cross-fitted Stage11 recipes**"))
display(stage11_table)
display(Markdown("**Stage12 integrity summary**"))
display(stage12_integrity)

### 5.10 Stage14 status and current-pipeline boundary

In [ ]:
stage14_summary = pd.DataFrame([
    ["Stage14 status", stage14_manifest["multitask_branch_status"]],
    ["Historical true joint multi-task implemented", stage14_manifest["historical_true_joint_multitask_implemented"]],
    ["Historical true joint multi-task evaluated", stage14_manifest["historical_true_joint_multitask_evaluated"]],
    ["Historical true joint multi-task promoted", stage14_manifest["historical_true_joint_multitask_promoted"]],
    ["Current pipeline jointly retrained", stage14_manifest["current_pipeline_is_jointly_retrained"]],
    ["Current pipeline mode", stage14_manifest["current_pipeline_mode"]],
    ["New Stage14 training performed", stage14_manifest["new_training_performed"]],
], columns=["Stage14 item", "Frozen value"])

display(stage14_summary)
display(Markdown("**Claim boundary**"))
print(stage14_manifest["claim_boundary"])

### 5.11 Acceptance and reproducibility evidence

In [ ]:
cuda_acceptance = load_json(EVIDENCE / "MSLesSeg_P10_T1_ACCEPTANCE.json")
release_acceptance = load_json(EVIDENCE / "NO_DRIVE_EVALUATOR_ACCEPTANCE.json")

acceptance = pd.DataFrame([
    [
        "P10_T1 CUDA end-to-end replay",
        cuda_acceptance["status"],
        cuda_acceptance["fold"],
        cuda_acceptance["geometry_matches"],
        cuda_acceptance["mask_matches"],
        cuda_acceptance["mismatched_voxels"],
        cuda_acceptance["scope"],
    ],
    [
        "Fresh-clone public-release execution",
        release_acceptance["status"],
        release_acceptance["resolved_fold"],
        "n/a",
        "n/a",
        "n/a",
        release_acceptance["scientific_scope"],
    ],
], columns=[
    "Evidence", "Status", "Fold", "Geometry match", "Mask match",
    "Mismatched voxels", "Scope",
])

display(acceptance)

print("Accepted P10_T1 prediction SHA-256:", cuda_acceptance["prediction_sha256"])
print("Public neural asset source:", release_acceptance["neural_assets"]["source"])
print("Original model Drive required:", release_acceptance["neural_assets"]["original_research_drive_required_for_neural_assets"])

### 5.12 Quantitative-results checklist

At this point the notebook has surfaced the complete frozen metric families committed in the public research package:

- Stage16: DSC, IoU, sensitivity, specificity, HD95 — aggregate, per fold, and all 93 cases.
- Stage13 classification: nested ROC-AUC/AP and fixed-OOF ROC-AUC, AP, Brier, accuracy, balanced accuracy, sensitivity, specificity, precision, F1, and both operating thresholds.
- Stage13 regression: nested RMSE/MAE/R²/Spearman and fixed-OOF MAE/RMSE/R²/Spearman.
- Development cohort counts, fold counts, Stage11 thresholds/minimum-volume recipes, Stage12 integrity, Stage14 status, Stage15 training configuration, Stage12–16 audit counts, and implementation acceptance evidence.

The patient-level section below is a separate inference demonstration and does not silently reuse ground truth.

## 6. Patient configuration

The default path is now a **zero-upload reproducibility demo**.

- `USE_BUNDLED_DEMO = True` uses the committed, attributed `MSLesSeg_P10_T1` FLAIR/T1/T2 triplet automatically.
- Set `USE_BUNDLED_DEMO = False` only when you want to test your own MRI triplet.
- For `MSLesSeg_P10_T1`, leave `FOLD = None`; the repository resolves its held-out fold automatically.
- For a case outside the frozen development registry, set `CASE_ID` and an explicit `FOLD` from 0 to 4.

An explicit fold for an unseen case is a **selected-fold research run**, not a validated new-patient ensemble.

In [ ]:
USE_BUNDLED_DEMO = True

CASE_ID = "MSLesSeg_P10_T1"
FOLD = None   # Known development case: automatic held-out-fold resolution.

print("USE_BUNDLED_DEMO:", USE_BUNDLED_DEMO)
print("CASE_ID:", CASE_ID)
print("FOLD:", FOLD)

## 7. Prepare FLAIR, T1 and T2

With the default `USE_BUNDLED_DEMO = True`, **there is nothing to upload**. The notebook reconstructs the attributed public demo MRI triplet already bundled with the repository and verifies each file by SHA-256.

If you change `USE_BUNDLED_DEMO = False`, this cell switches to the normal Colab file picker so you can supply your own co-registered FLAIR/T1/T2 NIfTI files.

The notebook never asks for or uses a ground-truth lesion mask.

In [ ]:
from pathlib import Path
import shutil, subprocess, sys

INPUT_DIR = Path("/content/patient_input")
if INPUT_DIR.exists():
    shutil.rmtree(INPUT_DIR)
INPUT_DIR.mkdir(parents=True)

if USE_BUNDLED_DEMO:
    if CASE_ID != "MSLesSeg_P10_T1":
        raise RuntimeError(
            "Bundled demo inputs are only defined for CASE_ID='MSLesSeg_P10_T1'. "
            "Set USE_BUNDLED_DEMO=False to use another patient."
        )
    subprocess.run(
        [
            sys.executable,
            "scripts/prepare_demo_inputs.py",
            "--output-dir",
            str(INPUT_DIR),
        ],
        check=True,
    )
else:
    from google.colab import files
    uploaded = files.upload()
    for name in uploaded:
        shutil.move(name, INPUT_DIR / name)

print("\nMRI input files:")
for p in sorted(INPUT_DIR.iterdir()):
    print(f"  {p.name}  ({p.stat().st_size:,} bytes)")

## 8. Resolve the three modalities and the frozen fold

The accepted MSLesSeg/nnU-Net convention is:
- `_0000` → FLAIR
- `_0001` → T1
- `_0002` → T2

The bundled demo is already prepared with this convention. For your own files, rename them to this convention before running this cell.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(REPO))

def exactly_one(suffix):
    matches = sorted(INPUT_DIR.glob(f"*{suffix}"))
    if len(matches) != 1:
        raise RuntimeError(
            f"Expected exactly one uploaded file ending {suffix}; found {len(matches)}: "
            + ", ".join(p.name for p in matches)
        )
    return matches[0]

FLAIR = exactly_one("_0000.nii.gz")
T1 = exactly_one("_0001.nii.gz")
T2 = exactly_one("_0002.nii.gz")

from graphms.patient import resolve_fold
SELECTED_FOLD, RUN_SCOPE = resolve_fold(CASE_ID, FOLD, REPO)

print("FLAIR:", FLAIR)
print("T1:   ", T1)
print("T2:   ", T2)
print("Resolved fold:", SELECTED_FOLD)
print("Run scope:", RUN_SCOPE)

## 9. Fetch and verify the required frozen neural assets

Only the required fold is downloaded from the public `assets-v1` GitHub Release.

Every file is checked for expected size and SHA-256 identity before it is admitted into the local asset lock.

In [ ]:
import subprocess, sys

subprocess.run(
    [
        sys.executable,
        "scripts/download_release_assets.py",
        "--folds",
        str(SELECTED_FOLD),
    ],
    check=True,
)

# Independent second verification through the normal runtime asset gate.
subprocess.run(
    [
        sys.executable,
        "scripts/setup_assets.py",
        "--folds",
        str(SELECTED_FOLD),
    ],
    check=True,
)

## 10. Run the complete patient pipeline

This invokes the repository's frozen patient-level inference entry point:

**FLAIR + T1 + T2 → frozen nnU-Net/ResEncM-250 → Stage5/6 graph features → Stage7 TrueGAT → GraphMS Hybrid → Stage11 → canonical Stage12 → frozen Stage13 → report/provenance.**

No training occurs here.

In [ ]:
import subprocess, sys, shutil
from pathlib import Path

OUTPUT_DIR = REPO / "outputs" / f"{CASE_ID}_colab_evaluator"

# Notebook convenience only: ensure this execution starts with a fresh output directory.
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

cmd = [
    sys.executable,
    "run_graphms.py",
    "--case-id", CASE_ID,
    "--flair", str(FLAIR),
    "--t1", str(T1),
    "--t2", str(T2),
    "--output", str(OUTPUT_DIR),
]
if FOLD is not None:
    cmd.extend(["--fold", str(FOLD)])

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

print("\nOutput directory:", OUTPUT_DIR)

## 11. Confirm completion and inspect the generated bundle

A valid completed run must contain `COMPLETE.json` with status `COMPLETE`.

In [ ]:
import json
from pathlib import Path

print("Generated outputs:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(" ", p.name)

complete = json.loads((OUTPUT_DIR / "COMPLETE.json").read_text())
assert complete.get("status") == "COMPLETE", complete

print("\n=== COMPLETE.json ===")
print(json.dumps(complete, indent=2))

## 12. Patient-specific Stage12 features and Stage13 research outputs

These values are generated by the live inference run. They are distinct from the frozen development-evaluation metrics shown above.

In [ ]:
import json
import pandas as pd
from IPython.display import display, Markdown

features = pd.read_csv(OUTPUT_DIR / "features.csv")
lesions = pd.read_csv(OUTPUT_DIR / "lesions.csv")
risk = json.loads((OUTPUT_DIR / "risk.json").read_text())

print("Stage12 feature table shape:", features.shape)

preferred_feature_columns = [
    "lesion_volume_mm3",
    "lesion_count",
    "largest_lesion_fraction",
    "lesion_volume_normalized",
    "periventricular_fraction_le3mm",
    "ventricle_distance_mean_mm",
    "atlas_brainstem_overlap_mm3",
    "atlas_cortical_overlap_mm3",
    "atlas_subcortical_overlap_mm3",
]
available = [c for c in preferred_feature_columns if c in features.columns]

display(Markdown("**Selected patient-level Stage12 features**"))
if available:
    display(features[available].T.rename(columns={0: "Value"}))
else:
    display(features.head())

display(Markdown("**Lesion-component table**"))
print("Lesion component count:", len(lesions))
display(lesions)

patient_risk = pd.DataFrame([
    ["EDSS≥4 probability", risk["edss_ge4_probability"]],
    ["Balanced threshold", risk["edss_ge4_balanced_threshold"]],
    ["Balanced prediction", risk["edss_ge4_balanced_prediction"]],
    ["High-sensitivity threshold", risk["edss_ge4_high_sensitivity_threshold"]],
    ["High-sensitivity prediction", risk["edss_ge4_high_sensitivity_prediction"]],
    ["Predicted EDSS", risk["predicted_edss"]],
    ["Classifier", risk["classifier"]],
    ["Regressor", risk["regressor"]],
    ["Feature set", risk["feature_set"]],
    ["Claim scope", risk["claim_scope"]],
], columns=["Patient-level Stage13 output", "Value"])

display(Markdown("**Patient-level Stage13 output summary**"))
display(patient_risk)

display(Markdown("**Complete risk.json**"))
print(json.dumps(risk, indent=2))

## 13. Display the segmentation overlay and HTML patient report

In [ ]:
from IPython.display import Image, HTML, display

print("=== Segmentation overlay ===")
display(Image(filename=str(OUTPUT_DIR / "overlay.png")))

print("\n=== Patient report ===")
display(HTML(filename=str(OUTPUT_DIR / "patient_report.html")))

## 14. Reproducibility check for the accepted demonstration case

For `MSLesSeg_P10_T1`, the repository contains an independent CUDA acceptance record. This cell compares the newly generated lesion-mask SHA-256 with the accepted prediction SHA.

For other patients, this exact reference comparison is intentionally skipped.

In [ ]:
import hashlib, json

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

if CASE_ID == "MSLesSeg_P10_T1":
    accepted = json.loads(
        (REPO / "evidence/acceptance/MSLesSeg_P10_T1_ACCEPTANCE.json").read_text()
    )
    current_hash = sha256(OUTPUT_DIR / "lesion_mask.nii.gz")
    expected_hash = accepted["prediction_sha256"]

    print("Current lesion-mask SHA-256: ", current_hash)
    print("Accepted prediction SHA-256:", expected_hash)
    print("Exact accepted-mask hash match:", current_hash == expected_hash)

    assert current_hash == expected_hash, "Generated mask does not match the accepted frozen prediction."
else:
    print("Exact accepted-mask comparison is only defined for MSLesSeg_P10_T1.")

### Frozen development metrics for the demonstration case

For `MSLesSeg_P10_T1`, these are the case-level metrics already committed in Stage16. They are displayed for scientific context only; they are **not recomputed by the live patient run**, because the live run does not load ground truth.

In [ ]:
if CASE_ID == "MSLesSeg_P10_T1":
    demo_reference = stage16_per_case.loc[
        stage16_per_case["case"] == CASE_ID,
        ["case", "fold", "DSC", "IoU", "Sensitivity", "Specificity", "HD95_mm", "TP", "FP", "FN", "TN"],
    ].copy()
    display(demo_reference)
else:
    print("Frozen case-level development metrics are displayed only for the bundled P10_T1 reproducibility case.")

## 15. Inspect provenance

This is where an evaluator can verify that the patient run did not use ground truth or perform new training.

In [ ]:
import json

provenance = json.loads((OUTPUT_DIR / "provenance.json").read_text())
print(json.dumps(provenance, indent=2))

assert provenance.get("ground_truth_used") is False
assert provenance.get("new_training_performed") is False

print("\nProvenance checks PASS: ground_truth_used=False, new_training_performed=False")

## 16. Download the complete output bundle

This packages the generated NIfTI, CSV, JSON, overlay, HTML report and provenance into one ZIP for inspection or handoff.

In [ ]:
import shutil
from google.colab import files

zip_base = Path("/content") / f"{CASE_ID}_GraphMS_outputs"
zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=OUTPUT_DIR)

print("Created:", zip_path)
files.download(zip_path)

---

## Interpretation

A successful execution demonstrates that the public GraphMS-Net repository can reproduce its frozen package checks, present the complete committed development metrics, obtain SHA-256-locked neural assets from the public release, execute the selected patient-level inference path, and produce the downstream research outputs with provenance.

The notebook intentionally separates **frozen evaluation evidence** from **live inference output**. Development ground truth is used only in the committed Stage16 evaluation artifacts; it is not supplied to the live patient-level inference run.

The reported results remain research results under the stated development-validation scope and are not presented as external clinical validation.